In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
from pint.errors import UnitStrippedWarning

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, ElectricVehicleBatteries_TV, BatteryMaterials, EvBatteryLinkModule
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.preprocessing import get_preprocessing_data_evbattery, get_preprocessing_data_evbattery_old

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_data = Path(path_base, "data", "raw")

# Old

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    

    factory = ModelFactory(
        [vhc_sector], complete_timeline
        ).add(GenericStocks, "vehicles"
        ).add(BatteryMaterials, "vehicles"
        ).add(GenericMaterials, "vehicles"
        )
    model = factory.finish()

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UnitStrippedWarning)
        model.simulate(simulation_timeline)
    all_output[scen_id] = model
    print(f"Finished {scen_id}")

list(model.vehicles)

# New

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output_2 = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_data, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_data, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_data,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    # elc_sector = get_preprocessing_data("electricity", path_data,
    #                                     climate_policy_scenario_dir, 
    #                                     circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector], complete_timeline
        ).add(GenericStocks, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(GenericMaterials, "vehicles"
        )
    model_2 = factory.finish()

    model_2.simulate(simulation_timeline)
    all_output_2[scen_id] = model_2
    print(f"Finished {scen_id}")

list(model_2.ev_battery)

# Compare

In [ ]:
# Plot ev_battery: materials over time
var = "inflow_battery_materials"
m1 = model.vehicles[var].sum(["battery", "Region", "Type"]).sel(Time=slice(1972,None))
m2 = model_2.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))

for mat in m1.material.values[:4]: #
    plt.plot(m1.Time, m1.sel(material=mat), label="old - " + str(mat))
    plt.plot(m2.time, m2.sel(material=mat), label="new - " + str(mat), linestyle=":", linewidth=2)
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kg)")
plt.legend()
plt.title(f"{var} in EV batteries")

In [ ]:
# Plot ev_battery: materials over time
var = "outflow_battery_materials"
m1 = model.vehicles[var].sum(["battery", "Region", "Type"]).sel(Time=slice(1972,None))
m2 = model_2.ev_battery[var].to_array().sum(["BatteryType", "Region", "Type"]).sel(time=slice(1972,None))

for mat in m1.material.values[:4]: #
    plt.plot(m1.Time, m1.sel(material=mat), label="m1 - " + str(mat))
    plt.plot(m2.time, m2.sel(material=mat), label="m2 - " + str(mat), linestyle=":", linewidth=2)
plt.xlabel("Year")
plt.ylabel(f"{var} in EV batteries (kg)")
plt.legend()
plt.title(f"{var} in EV batteries")